# Supervised Classification

Random Forest · Gradient Boosting · Logistic Regression
Trained only on labeled data, stratified train/test split.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.data_loader import load_all
from src.preprocessing import prepare_supervised
from src.models import random_forest, gradient_boosting, logistic_regression, predict_anomaly, anomaly_scores
from src.evaluation import evaluate, plot_confusion_matrix, plot_pr_curve, plot_roc_curve

df, edges = load_all()
X_train, X_test, y_train, y_test, feat_cols, scaler = prepare_supervised(df)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Train illicit rate: {y_train.mean():.3f}')

## Random Forest

In [ ]:
rf = random_forest(X_train, y_train)
y_pred_rf = predict_anomaly(rf, X_test)
y_score_rf = anomaly_scores(rf, X_test)

res_rf = evaluate(y_test, y_pred_rf, y_score_rf, name='RandomForest')
plot_confusion_matrix(y_test, y_pred_rf, name='RandomForest')

## Gradient Boosting

In [ ]:
gb = gradient_boosting(X_train, y_train)
y_pred_gb = predict_anomaly(gb, X_test)
y_score_gb = anomaly_scores(gb, X_test)

res_gb = evaluate(y_test, y_pred_gb, y_score_gb, name='GradientBoosting')
plot_confusion_matrix(y_test, y_pred_gb, name='GradientBoosting')

## Logistic Regression (baseline)

In [ ]:
lr = logistic_regression(X_train, y_train)
y_pred_lr = predict_anomaly(lr, X_test)
y_score_lr = anomaly_scores(lr, X_test)

res_lr = evaluate(y_test, y_pred_lr, y_score_lr, name='LogisticRegression')
plot_confusion_matrix(y_test, y_pred_lr, name='LogisticRegression')

## Comparison

In [ ]:
import pandas as pd
results = pd.DataFrame([res_rf, res_gb, res_lr]).set_index('name')
print(results.sort_values('f1', ascending=False))

plot_pr_curve(y_test, {'RandomForest': y_score_rf, 'GradientBoosting': y_score_gb, 'LogisticRegression': y_score_lr})
plot_roc_curve(y_test, {'RandomForest': y_score_rf, 'GradientBoosting': y_score_gb, 'LogisticRegression': y_score_lr})

## Feature Importance (Random Forest)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

importances = rf.feature_importances_
top_idx = np.argsort(importances)[::-1][:20]
top_feats = [feat_cols[i] for i in top_idx]
top_vals  = importances[top_idx]

plt.figure(figsize=(10, 5))
plt.bar(top_feats, top_vals)
plt.xticks(rotation=45, ha='right')
plt.title('Top 20 feature importances — Random Forest')
plt.tight_layout()
plt.show()